# Phase 1 Task B — Ablation 3.1: 90° Rotation — MobileNetV3-Large

**Experiment:** Discrete 90° rotation augmentation (0°, 90°, 180°, 270°)
**Model:** MobileNetV3-Large
**Seed:** 42
**Batch:** 64
**Output:** wandb `woodbench-taskB`, output dir `data/experiments/taskB_90rotation/mobilenetv3_large/`

So sánh với baseline M1 (random ±30° rotation) để đánh giá hiệu quả của discrete rotation cho ảnh isotropic.

In [2]:
# ── Cell 1: Root Detection ────────────────────────────────────
from pathlib import Path
import sys, os

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    NOTEBOOK = Path("__file__").resolve()
    ROOT = NOTEBOOK
    for _ in range(10):
        ROOT = ROOT.parent
        if (ROOT / "pyproject.toml").exists() or (ROOT / ".git").exists(): break

sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
print(f"[ROOT] {ROOT}")

[ROOT] /workspace/woodbench


In [3]:
# ── Cell 2: Environment Setup ─────────────────────────────────
from dotenv import load_dotenv
dotenv_path = ROOT / ".env"
load_dotenv(dotenv_path)
print(f"[.env] Loaded from {dotenv_path}")

import subprocess, sys
# for pkg in ["timm", "albumentations", "scikit-learn", "seaborn", "wandb", "ptflops", "python-dotenv"]:
#     try: __import__(pkg.replace("-", "_"))
#     except: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

os.environ.setdefault("WANDB_API_KEY", os.getenv("WANDB_API_KEY", ""))
os.environ.setdefault("HF_TOKEN", os.getenv("HF_TOKEN", ""))
print(f"[wandb] API key: {'set' if os.getenv('WANDB_API_KEY') else 'MISSING'}")
print(f"[HF] token: {'set' if os.getenv('HF_TOKEN') else 'MISSING'}")

[.env] Loaded from /workspace/woodbench/.env
[wandb] API key: set
[HF] token: set


In [4]:
# ── Cell 3: Data Download ─────────────────────────────────────
HF_REPO = "ny0g507h3p/wood-task-B"
DATA_ROOT = ROOT / "data" / "task-B"

if not (DATA_ROOT / "train").exists():
    print(f"[HF] Downloading dataset to {DATA_ROOT}...")
    from huggingface_hub import login, snapshot_download
    token = os.getenv("HF_TOKEN", "")
    if token:
        login(token=token)
        print("[HF] Logged in")
    else:
        print("[HF] No token — trying public access")
    snapshot_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        local_dir=str(DATA_ROOT),
        ignore_patterns=["*.md", "README*"],
    )
    print("[HF] Download complete")
else:
    print(f"[Data] Found at {DATA_ROOT}")

# Verify
train_dir = DATA_ROOT / "train"
if not train_dir.exists():
    raise FileNotFoundError(f"[ERROR] train/ not found in {DATA_ROOT}. Dataset download failed.")
classes = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])
print(f"[Data] {len(classes)} classes: {classes[:3]}...")
print(f"[Data] Train samples: {sum(1 for _ in train_dir.rglob('*.jpg')) + sum(1 for _ in train_dir.rglob('*.png'))}")

[Data] Found at /workspace/woodbench/data/task-B
[Data] 20 classes: ['Araucaria araucana', 'Callitris columellaris', 'Calocedrus rupestris']...
[Data] Train samples: 3973


In [5]:
# ── Cell 4: DataLoaders (use_90rotation=True) ─────────────────
from src.data import get_dataloaders
tl, vl, testl = get_dataloaders(
    str(DATA_ROOT), batch_size=64, num_workers=4, task="taskB",
    use_90rotation=True,   # ← ABLATION: discrete 0/90/180/270
)
print(f"[Data] Train={len(tl.dataset)}, Val={len(vl.dataset)}, Test={len(testl.dataset)}")
print(f"[Data] Classes={len(tl.dataset.classes)}")
print(f"[Data] Augmentation: use_90rotation=True (discrete 0°/90°/180°/270°)")

/venv/main/lib/python3.12/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info Remote end closed connection without response
  data = fetch_version_info()


[Data] Train=3973, Val=933, Test=924
[Data] Classes=20
[Data] Augmentation: use_90rotation=True (discrete 0°/90°/180°/270°)


In [6]:
# ── Cell 5: Imports & Device ──────────────────────────────────
import wandb, torch, numpy as np, json
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from src.models import create_model
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function, WandbTracker
from src.utils import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device] {DEVICE}")

[Device] cuda


In [7]:
# ── Cell 6: train_one() — with WandbTracker + Test Evaluation ─

def train_one(backbone, seed, data_root, batch_size=64, epochs=50):
    """Train one model with 90° rotation augmentation.
    
    Improvements over M1 notebooks:
    1. WandbTracker passed to engine.fit() → epoch-level metrics logged
    2. Test evaluation after training → test_metrics.json saved
    3. use_90rotation=True in dataloaders
    """
    set_seed(seed)
    # Output to taskB_90rotation/ (separate from baseline)
    output_dir = ROOT / "data" / "experiments" / "taskB_90rotation" / backbone
    output_dir.mkdir(parents=True, exist_ok=True)

    # DataLoaders with 90° rotation
    train_loader, val_loader, test_loader = get_dataloaders(
        data_root=data_root, batch_size=batch_size, num_workers=4, task="taskB",
        use_90rotation=True,
    )
    num_classes = len(train_loader.dataset.classes)
    class_names = train_loader.dataset.classes
    print(f"[{backbone} s{seed}] train={len(train_loader.dataset)}, classes={num_classes}")

    # WandbTracker (logs epoch-level metrics automatically)
    config = {
        "backbone": backbone, "seed": seed, "epochs": epochs,
        "batch_size": batch_size, "num_classes": num_classes,
        "lr_head": 1e-4, "lr_backbone": 1e-5,
        "label_smoothing": 0.1, "scheduler": "cosine", "amp": True,
        "experiment": "taskB_90rotation",
        "augmentation": "discrete_90 (0/90/180/270)",
    }
    tracker = WandbTracker.start(
        project="woodbench-taskB", name=f"{backbone}_90rot_s{seed}",
        config=config, dir_path=str(output_dir),
        tags=["ablation", "90rotation", backbone, f"seed{seed}"],
    )

    model = create_model(backbone, num_classes=num_classes, pretrained=True)
    optimizer = create_optimizer(model, lr=1e-4, weight_decay=1e-4, backbone_lr=1e-5)
    scheduler = create_scheduler(optimizer, epochs=epochs, scheduler_type="cosine")
    criterion = create_loss_function("label_smoothing", label_smoothing=0.1)

    engine = TrainingEngine(
        model=model, train_loader=train_loader, val_loader=val_loader,
        optimizer=optimizer, scheduler=scheduler, criterion=criterion,
        device=DEVICE, use_amp=True,
    )
    result = engine.fit(
        epochs=epochs, output_dir=output_dir,
        name=f"{backbone}_seed{seed}", save_best_only=True,
        early_stopping_patience=10, tracker=tracker,
    )

    # ── Test Evaluation ──────────────────────────────────────────
    print(f"\n[{backbone} s{seed}] Evaluating on test set...")
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(DEVICE)
            preds = model(images).argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    prec = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    rec  = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1_per = f1_score(all_labels, all_preds, average=None, zero_division=0)

    print(f"[{backbone} s{seed}] TEST Acc={acc*100:.2f}%  F1={f1*100:.2f}%  Prec={prec*100:.2f}%  Rec={rec*100:.2f}%")

    # Log test metrics to wandb
    if tracker:
        tracker.log({
            "test/accuracy": acc, "test/f1_macro": f1,
            "test/precision_macro": prec, "test/recall_macro": rec,
            **{f"test/f1/{cn}": float(fi) for cn, fi in zip(class_names, f1_per)},
        }, step=result["epochs_trained"])

    # Save test_metrics.json
    test_metrics = {
        "backbone": backbone, "seed": seed,
        "experiment": "taskB_90rotation",
        "best_val_acc": result["best_val_acc"],
        "best_epoch": result["best_epoch"],
        "epochs_trained": result["epochs_trained"],
        "stopped_early": result["stopped_early"],
        "test_accuracy": float(acc),
        "test_f1_macro": float(f1),
        "test_precision_macro": float(prec),
        "test_recall_macro": float(rec),
        "test_f1_per_class": {cn: float(fi) for cn, fi in zip(class_names, f1_per)},
    }
    with open(output_dir / "test_metrics.json", "w") as fp:
        json.dump(test_metrics, fp, indent=2)
    print(f"  → Saved: {output_dir / 'test_metrics.json'}")

    if tracker:
        tracker.finish()

    print(f"[{backbone} s{seed}] val_acc={result['best_val_acc']:.2f}%")
    return {**result, "test_metrics": test_metrics}

In [8]:
# ── Cell 7: Run Training (seed 42 only) ──────────────────────
SEED = 42

print(f"\n{'='*60}")
print(f"Ablation 3.1: 90° Rotation — MobileNetV3-Large | seed={SEED}")
print(f"{'='*60}")

result = train_one("mobilenetv3_large", SEED, str(DATA_ROOT), batch_size=64, epochs=50)


Ablation 3.1: 90° Rotation — MobileNetV3-Large | seed=42


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


[mobilenetv3_large s42] train=3973, classes=20


wandb: Currently logged in as: ny0g507h3p (ny0g507h3p_) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

Epoch 1/50 | Train Loss: 3.0021, Acc: 6.65% | Val Loss: 2.9986, Acc: 7.82% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to mobilenetv3_large_seed42_best.pth (Val Acc: 7.82%)


... (32 outputs left out)

Epoch 34/50 | Train Loss: 0.6708, Acc: 99.34% | Val Loss: 0.6630, Acc: 99.57% | LR: 2.81e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 34.

[mobilenetv3_large s42] Evaluating on test set...


[mobilenetv3_large s42] TEST Acc=99.46%  F1=99.46%  Prec=99.48%  Rec=99.45%
  → Saved: /workspace/woodbench/data/experiments/taskB_90rotation/mobilenetv3_large/test_metrics.json


lr,▁▃▅▆███████▇▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂
test/accuracy,▁
test/f1/Araucaria araucana,▁
test/f1/Callitris columellaris,▁
test/f1/Calocedrus rupestris,▁
test/f1/Calocedrus sp,▁
test/f1/Chamaecyparis obtusa,▁
test/f1/Cryptomeria japonica,▁
test/f1/Cunninghamia lanceolata,▁
test/f1/Cupressus funebris (Cupressus pendula),▁
+19,...


[mobilenetv3_large s42] val_acc=99.68%


In [9]:
# ── Cell 8: Summary ───────────────────────────────────────────
print("\n" + "="*60)
print("Ablation 3.1: 90° Rotation — MobileNetV3-Large — Summary")
print("="*60)
print(f"  Seed:           {SEED}")
print(f"  Val Acc (best):  {result['best_val_acc']:.2f}%")
print(f"  Best Epoch:      {result['best_epoch']}")
print(f"  Epochs Trained:  {result['epochs_trained']}")
print(f"  Early Stopped:   {result['stopped_early']}")
print(f"  ---")
tm = result['test_metrics']
print(f"  Test Accuracy:   {tm['test_accuracy']*100:.2f}%")
print(f"  Test F1 (macro): {tm['test_f1_macro']*100:.2f}%")
print(f"  Test Precision:  {tm['test_precision_macro']*100:.2f}%")
print(f"  Test Recall:     {tm['test_recall_macro']*100:.2f}%")
print(f"\n  Output: data/experiments/taskB_90rotation/mobilenetv3_large/")


Ablation 3.1: 90° Rotation — MobileNetV3-Large — Summary
  Seed:           42
  Val Acc (best):  99.68%
  Best Epoch:      24
  Epochs Trained:  34
  Early Stopped:   True
  ---
  Test Accuracy:   99.46%
  Test F1 (macro): 99.46%
  Test Precision:  99.48%
  Test Recall:     99.45%

  Output: data/experiments/taskB_90rotation/mobilenetv3_large/
